In [10]:
from IPython.core.display import HTML
import json
import io

import pandas as pd
import numpy as np
import jinja2
import datasets
from sklearn.model_selection import GroupShuffleSplit
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env")
client = OpenAI()

## Explore old MultiRC Dataset

- This dataset has GPT-generated reference answers.
- It uses the questions and candidate answers from MultiRC.
- MultiRC includes multiple correct and multiple incorrect answers for each question.
- There are multiple questions for each passage.

PROBLEMS: The generated reference answers are pretty poor. Will need to re-generate with a more modern LLM.

In [4]:
df = pd.read_csv(
    "../../data/multirc.csv", index_col=0
)
for i, (source_text, group) in enumerate(df.groupby("source_text")):
    if i < 2:
        continue
    display(HTML(source_text))
    for (question, gpt_answer), subgroup in group.groupby(["question_text", "gpt_answers"]):
        print("Question:", question)
        print("GPT Answer:", gpt_answer.strip().removeprefix("Answer:"))
        for sample in subgroup.itertuples():
            # print("GPT Answer:", sample.gpt_answers.strip().removeprefix("Answer:"))
            print("-"*80)
            print(f"{'CORRECT' if sample.is_correct else 'WRONG'} Answer:", sample.answer_text)
            # print("Correct:", sample.is_correct)
        print("="*80)
    break

Question: At what time should Christians fast?
GPT Answer: Christians should fast at any time that they feel sadness or longing for the coming of Christ in glory.
--------------------------------------------------------------------------------
CORRECT Answer: The time when the bridegroom was taken away
--------------------------------------------------------------------------------
CORRECT Answer: The bridegroom should be taken from us
--------------------------------------------------------------------------------
WRONG Answer: The bridegroom should be bring to us
Question: Did Christ seem to leave the world?
GPT Answer: No, Christ did not seem to leave the world. He promised to return in a little while, and hundreds of years later, the early Christians were still waiting for His coming.
--------------------------------------------------------------------------------
WRONG Answer: No
--------------------------------------------------------------------------------
WRONG Answer: Christ


## Generate new reference answers

In [41]:
# Already partitioned without overlap in the source_text / passage
dd = datasets.DatasetDict.load_from_disk("../../data/multirc_contrastive_pairs.hf")

df = pd.concat(
    [ds.to_pandas() for ds in dd.values()],
    keys=dd.keys()
).reset_index(level=0, names="split")
df

,split,provided_split,passage,question,answer,label,passage_id,question_id,answer_id
0,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 17,0,1,26,168
1,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 29,1,1,26,169
2,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,October 29,0,1,26,170
3,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,October 17,0,1,26,171
4,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 17,0,1,26,172
...,...,...,...,...,...,...,...,...,...
5531,test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,The same image as you,0,79,917,4669
5532,test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,Your image is reversed and looks just like you,1,79,917,4670
5533,test,validation,You have seen your own reflection in a mirror....,What causes the image in a mirror reflection t...,Light rays strike flat shiny surfaces and are ...,1,79,918,4671
5534,test,validation,You have seen your own reflection in a mirror....,What causes the image in a mirror reflection t...,The reflection reversed because the mirror is ...,0,79,918,4672


In [43]:
questions_df = df[["split", "provided_split", "passage", "question", "passage_id", "question_id"]].drop_duplicates()
questions_df

,split,provided_split,passage,question,passage_id,question_id
0,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,1,26
5,train,train,"The rally took place on October 17, the shooti...",Who was president of the NRA on February 29?,1,27
11,train,train,"The rally took place on October 17, the shooti...",What organization had a rally in Flint on Octo...,1,28
18,train,train,"The rally took place on October 17, the shooti...","How many times does Moore use the ""from my col...",1,29
21,train,train,I'm here to tell you the story of a robot name...,What did Carl need before going to the lab,2,30
...,...,...,...,...,...,...
5512,test,validation,You have seen your own reflection in a mirror....,What is the only difference between a reflecti...,79,914
5516,test,validation,You have seen your own reflection in a mirror....,What happens when you look at your reflection?,79,915
5523,test,validation,You have seen your own reflection in a mirror....,What will you notice about your reflection whe...,79,916
5528,test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,79,917


In [44]:
questions_df["id"] = questions_df["provided_split"] + "_" + questions_df["passage_id"].astype(str) + "_" + questions_df["question_id"].astype(str)

# Ensure no duplicate IDs
questions_df[questions_df.id.duplicated(keep=False)].sort_values("question_id")

,split,provided_split,passage,question,passage_id,question_id,id


In [45]:
environment = jinja2.Environment(loader=jinja2.FileSystemLoader("../../prompts/"))
template = environment.get_template("reference-answer-generate.jinja2")

print(template.render(questions_df.sample(1).to_dict(orient="records")[0]))

You are an expert educational assessment designer. Your task is to write a reference answer for the following reading comprehension question. Your answer will be used to score student responses to the question. Both the reading passage and the reading comprehension question are provided.

INSTRUCTIONS:
Generate a reference answer for the following reading comprehension question.

READING PASSAGE:
"""The inhabited history of the Las Vegas Valley stretches to 23,000 b.c. , when much of the area was covered by a prehistoric lake. During this period, the indigenous people lived in caves, hunting the mammals that gathered at the shoreline. The landscape of the valley changed dramatically over the next 200 centuries. The glaciers feeding the lake melted away and the lake evaporated. Fossils tell an obscure story of man's slow and sporadic development. Around 3000 b.c. , native Archaic Indians began to develop a lasting hunting and gathering culture. By this time, the valley was in much the s

In [46]:
sample = questions_df.sample(1)
content = template.render(sample.to_dict(orient="records")[0])
print(content)

response = client.chat.completions.create(
    model="gpt-5-mini-2025-08-07",
    messages=[{"role": "user", "content": content}],
    response_format={  # {"answer": "yes"}
        "type": "json_schema",
        "json_schema": {
            "name": "reference_answer_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "answer": {
                        "description": "A reference answer for the constructed response item",
                        "type": "string",
                    },
                    "additionalProperties": False,
                },
            },
        },
    },
)

response_dict_string = response.choices[0].message.content
response_dict = json.loads(response_dict_string)
answer = response_dict["answer"]
print(f"\n\nGPT Reference Answer: {answer}")

You are an expert educational assessment designer. Your task is to write a reference answer for the following reading comprehension question. Your answer will be used to score student responses to the question. Both the reading passage and the reading comprehension question are provided.

INSTRUCTIONS:
Generate a reference answer for the following reading comprehension question.

READING PASSAGE:
"""We need natural resources for just about everything we do. We need them for food and clothing, for building materials and energy. We even need them to have fun. Table 2.1 gives examples of how we use natural resources. Can you think of other ways we use natural resources? Use Vehicles Resources Rubber for tires from rubber trees Steel frames and other metal parts from minerals such as iron Example iron ore Electronics Plastic cases from petroleum prod- ucts Glass screens from minerals such as lead lead ore Use Homes Resources Nails from minerals such as iron Timber from trees Example spruce

## Test LLM Response

In [47]:
def format_request(custom_id, prompt, model_name="gpt-5-mini-2025-08-07"):
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model_name,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "reference_answer_schema",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "answer": {
                                "description": "A reference answer for the constructed response item",
                                "type": "string",
                            },
                            "additionalProperties": False,
                        },
                    },
                },
            },
        },
    }


batch_input = pd.DataFrame(
    [
        format_request(
            row.id,
            template.render(**row._asdict()),
        )
        for row in questions_df.itertuples()
    ]
)

batch_input.to_json(
    "../../bin/multirc_batch_input_for_aqg.jsonl",
    orient="records",
    lines=True,
)

display(batch_input)
print(batch_input.iloc[0]["body"]["messages"][0]["content"])

,custom_id,method,url,body
0,train_1_26,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
1,train_1_27,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
2,train_1_28,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
3,train_1_29,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
4,train_2_30,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
...,...,...,...,...
6079,validation_79_914,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
6080,validation_79_915,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
6081,validation_79_916,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."
6082,validation_79_917,POST,/v1/chat/completions,"{'model': 'gpt-5-mini-2025-08-07', 'messages':..."


You are an expert educational assessment designer. Your task is to write a reference answer for the following reading comprehension question. Your answer will be used to score student responses to the question. Both the reading passage and the reading comprehension question are provided.

INSTRUCTIONS:
Generate a reference answer for the following reading comprehension question.

READING PASSAGE:
"""The rally took place on October 17, the shooting on February 29. Again, standard filmmaking techniques are interpreted as smooth distortion: "Moore works by depriving you of context and guiding your mind to fill the vacuum -- with completely false ideas. It is brilliantly, if unethically, done." As noted above, the "from my cold dead hands" part is simply Moore's way to introduce Heston. Did anyone but Moore's critics view it as anything else? He certainly does not "attribute it to a speech where it was not uttered" and, as noted above, doing so twice would make no sense whatsoever if Moore

In [48]:
batch_input_file = client.files.create(
    file=open("../../bin/multirc_batch_input_for_aqg.jsonl", "rb"),
    purpose="batch",
)
batch_input_file_id = batch_input_file.id
batch_details = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": "Question Generation MultiRC 1"},
)
print(f"batch-id:\n{batch_details.id}\n")
print(batch_details)

batch-id:
batch_68f29ec09da081909f8956150592a9d3

Batch(id='batch_68f29ec09da081909f8956150592a9d3', completion_window='24h', created_at=1760730816, endpoint='/v1/chat/completions', input_file_id='file-SKqTrErZCnFTHuQ1y1eTwm', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1760817216, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'Question Generation MultiRC 1'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), model=None, usage={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens_details': {'reasoning_tokens': 0}})


In [73]:
batch = client.batches.retrieve("batch_68f29ec09da081909f8956150592a9d3")
print(batch)
output_file_id = batch.output_file_id
print(output_file_id)
if output_file_id:
    print(output_file_id.split("-")[1])  # Just the id part for easy copy/paste

Batch(id='batch_68f29ec09da081909f8956150592a9d3', completion_window='24h', created_at=1760730816, endpoint='/v1/chat/completions', input_file_id='file-SKqTrErZCnFTHuQ1y1eTwm', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1760732287, error_file_id=None, errors=None, expired_at=None, expires_at=1760817216, failed_at=None, finalizing_at=1760731936, in_progress_at=1760730818, metadata={'description': 'Question Generation MultiRC 1'}, output_file_id='file-7NAXEJqsgCAzNjhj92KDqV', request_counts=BatchRequestCounts(completed=6084, failed=0, total=6084), model='gpt-5-mini-2025-08-07', usage={'input_tokens': 2763068, 'output_tokens': 1233302, 'total_tokens': 3996370, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens_details': {'reasoning_tokens': 972416}})
file-7NAXEJqsgCAzNjhj92KDqV
7NAXEJqsgCAzNjhj92KDqV


In [74]:
file_response = client.files.content("file-7NAXEJqsgCAzNjhj92KDqV")

In [94]:
results = pd.read_json(io.StringIO(file_response.text), lines=True)

In [95]:
results = results.set_index("custom_id")

def extract_response(response_dict):
    content = response_dict["body"]["choices"][0]["message"]["content"]
    return json.loads(content)["answer"]

results["gpt_reference_answer"] = results["response"].apply(extract_response)
results = results.drop(columns=["id", "custom_id", "error", "response"])
results

,provided_split,passage_id,question_id,gpt_reference_answer
0,train,1,26,She was shot on February 29.
1,train,1,27,Charlton Heston
2,train,1,28,The National Rifle Association (NRA).
3,train,1,29,Once.
4,train,2,30,He needed replacement parts — specifically a t...
...,...,...,...,...
6079,validation,79,914,The reflection is reversed — left and right ar...
6080,validation,79,915,You see a person who looks like you standing o...
6081,validation,79,916,You will notice the reflection waves with the ...
6082,validation,79,917,The image of the sign above (the sign’s reflec...


In [96]:
# Join with original dataframe
merged_df = pd.merge(df, results, on=["provided_split", "passage_id", "question_id"], how="inner")
merged_df

,split,provided_split,passage,question,answer,label,passage_id,question_id,answer_id,gpt_reference_answer
0,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 17,0,1,26,168,She was shot on February 29.
1,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 29,1,1,26,169,She was shot on February 29.
2,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,October 29,0,1,26,170,She was shot on February 29.
3,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,October 17,0,1,26,171,She was shot on February 29.
4,train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 17,0,1,26,172,She was shot on February 29.
...,...,...,...,...,...,...,...,...,...,...
32086,test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,The same image as you,0,79,917,4669,The image of the sign above (the sign’s reflec...
32087,test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,Your image is reversed and looks just like you,1,79,917,4670,The image of the sign above (the sign’s reflec...
32088,test,validation,You have seen your own reflection in a mirror....,What causes the image in a mirror reflection t...,Light rays strike flat shiny surfaces and are ...,1,79,918,4671,"Because light rays strike the flat, shiny surf..."
32089,test,validation,You have seen your own reflection in a mirror....,What causes the image in a mirror reflection t...,The reflection reversed because the mirror is ...,0,79,918,4672,"Because light rays strike the flat, shiny surf..."


In [98]:
merged_df.to_csv("../../data/multirc-data-w-gpt5-reference-answers.csv")